# 05 — Manual FreeCAD FEA Results Entry

Guide the user through FreeCAD FEM + CalculiX, collect the real results, and write the manual FEA report.


In [ ]:
from pathlib import Path
import json
import logging
import os
import sys

import yaml
from IPython.display import Image, Markdown, display

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "code_base").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
MODULE_ROOT = PROJECT_ROOT / "code_base" / "fea_cad_one_sample"
if str(MODULE_ROOT) not in sys.path:
    sys.path.insert(0, str(MODULE_ROOT))

API_ENV_PATH = MODULE_ROOT / "src" / "api.env"


def _load_api_env(path: Path) -> dict[str, str]:
    env_values: dict[str, str] = {}
    if not path.exists():
        return env_values
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip("'").strip('"')
        if key and value:
            env_values[key] = value
            os.environ.setdefault(key, value)
    return env_values


api_env_values = _load_api_env(API_ENV_PATH)

from src import interfaces as api

logging.basicConfig(level=logging.INFO, format="%(name)s | %(levelname)s | %(message)s")

FIXED_SAMPLE_PATH = MODULE_ROOT / "experiment_config" / "fixed_sample.yaml"
OUTPUT_ROOT = MODULE_ROOT / "outputs"
sample_config = yaml.safe_load(FIXED_SAMPLE_PATH.read_text(encoding="utf-8"))
sample_id = str(sample_config["sample_id"])
selection_source = str(sample_config.get("load_source", "dataset"))
state_b_dir = OUTPUT_ROOT / f"sample_{sample_id}" / "02_fea_constrained_revision"
manual_dir = OUTPUT_ROOT / f"sample_{sample_id}" / "04_manual_freecad_fea"
manual_dir.mkdir(parents=True, exist_ok=True)
screenshots_dir = manual_dir / "screenshots"
screenshots_dir.mkdir(parents=True, exist_ok=True)

state_b_step_path = state_b_dir / "fea_revision.step"
load_case_path = state_b_dir / "load_case.json"
instructions_path = manual_dir / "freecad_manual_instructions.md"
report_path = manual_dir / "fea_report.json"

print("[STAGE] Setup")
print(f"  → sample id : {sample_id}")
print(f"  → state B   : {state_b_step_path}")
print(f"  → manual dir: {manual_dir}")
print(f"  → report    : {report_path}")
print(f"  → api.env   : {API_ENV_PATH}")
print(f"  → api.env keys: {sorted(api_env_values)}")


In [6]:
print("[STAGE] Generate FreeCAD instructions")
assert state_b_step_path.exists(), f"missing State B STEP: {state_b_step_path}"
load_case = api.LoadCase(**json.loads(load_case_path.read_text(encoding="utf-8")))
instructions_written = api.write_freecad_instructions(sample_id, state_b_step_path, load_case, instructions_path, force=True)
display(Markdown(instructions_written.read_text(encoding="utf-8")))
print(f"  → instructions : {instructions_written}")
print("  → expected screenshots:")
for name in ["mesh.png", "fixed_region.png", "load_region.png", "von_mises.png", "displacement.png"]:
    print(f"    - {screenshots_dir / name}")


src.fea.freecad_manual_instructions | INFO | write_freecad_instructions | start | sample_id=00689964 | step_path=/Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/CAD-Physics/code_base/fea_cad_one_sample/outputs/sample_00689964/02_fea_constrained_revision/fea_revision.step | output_path=/Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/CAD-Physics/code_base/fea_cad_one_sample/outputs/sample_00689964/04_manual_freecad_fea/freecad_manual_instructions.md | force=True
src.fea.freecad_manual_instructions | INFO | write_freecad_instructions | done | sample_id=00689964 | output_path=/Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/CAD-Physics/code_base/fea_cad_one_sample/outputs/sample_00689964/04_manual_freecad_fea/freecad_manual_instructions.md


[STAGE] Generate FreeCAD instructions


# Manual FreeCAD FEM Instructions

Sample ID: 00689964
STEP file: /Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/CAD-Physics/code_base/fea_cad_one_sample/outputs/sample_00689964/02_fea_constrained_revision/fea_revision.step
Solver: FreeCAD FEM + CalculiX

## Load Case Summary

- Material: Aluminum 6061-T6
- Young's modulus: 68900000000 Pa
- Poisson's ratio: 0.33
- Yield strength: 276000000 Pa
- Fixed/support region: User-defined or model-inferred fixed support region
- Load region: User-defined or model-inferred load application region
- Applied load: 200 N direction [0, 0, -1]
- Max displacement target: 1.0 mm
- Required safety factor: 2.0
- Max von Mises target: 138000000 Pa

## Manual Steps

1. Open FreeCAD.
2. Import fea_revision.step.
3. Switch to the FEM workbench.
4. Create a new analysis.
5. Assign material: Aluminum 6061-T6.
6. Set Young's modulus, Poisson's ratio, and yield strength to the values above.
7. Add a fixed constraint to the User-defined or model-inferred fixed support region.
8. Add a force constraint: 200 N downward on the User-defined or model-inferred load application region.
9. Create the mesh using Gmsh or Netgen.
10. Run CalculiX manually.
11. Open the results.
12. Save screenshots for:
   - mesh.png
   - fixed_region.png
   - load_region.png
   - von_mises.png
   - displacement.png
13. Record the following results:
   - max von Mises stress
   - max displacement
   - solver success/failure
   - visible stress hotspot location

## macOS Note

If FreeCAD does not find CalculiX automatically, set the CalculiX binary path in FreeCAD preferences.
Do not require automated CalculiX execution in this prototype.


  → instructions : /Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/CAD-Physics/code_base/fea_cad_one_sample/outputs/sample_00689964/04_manual_freecad_fea/freecad_manual_instructions.md
  → expected screenshots:
    - /Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/CAD-Physics/code_base/fea_cad_one_sample/outputs/sample_00689964/04_manual_freecad_fea/screenshots/mesh.png
    - /Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/CAD-Physics/code_base/fea_cad_one_sample/outputs/sample_00689964/04_manual_freecad_fea/screenshots/fixed_region.png
    - /Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/CAD-Physics/code_base/fea_cad_one_sample/outputs/sample_00689964/04_manual_freecad_fea/screenshots/load_region.png
    - /Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/CAD-Physics/code_base/fea_cad_one_sample/outputs/sample_00689964/04_manual_free

In [7]:
print("[STAGE] Editable manual report template")
template = api.write_manual_fea_report_template(sample_id, report_path, force=True)
manual_report = {
    "sample_id": sample_id,
    "solver": template.solver,
    "manual_run": True,
    "max_von_mises_pa": None,
    "max_displacement_mm": None,
    "yield_strength_pa": template.yield_strength_pa,
    "required_safety_factor": template.required_safety_factor,
    "computed_safety_factor": None,
    "passes_stress": None,
    "passes_displacement": None,
    "overall_pass": None,
    "stress_hotspot_description": "",
    "notes": [],
}
display(Markdown("## Edit this JSON-like template after running FreeCAD"))
display(Markdown(f"```json\n{json.dumps(manual_report, indent=2)}\n```"))
print(f"  → template file : {report_path}")
assert report_path.exists()


src.fea.manual_report | INFO | write_manual_fea_report_template | start | sample_id=00689964 | output_path=/Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/CAD-Physics/code_base/fea_cad_one_sample/outputs/sample_00689964/04_manual_freecad_fea/fea_report.json | force=True
src.fea.manual_report | INFO | write_manual_fea_report_template | done | sample_id=00689964 | output_path=/Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/CAD-Physics/code_base/fea_cad_one_sample/outputs/sample_00689964/04_manual_freecad_fea/fea_report.json


[STAGE] Editable manual report template


## Edit this JSON-like template after running FreeCAD

```json
{
  "sample_id": "00689964",
  "solver": "FreeCAD FEM + CalculiX",
  "manual_run": true,
  "max_von_mises_pa": null,
  "max_displacement_mm": null,
  "yield_strength_pa": 276000000,
  "required_safety_factor": 2.0,
  "computed_safety_factor": null,
  "passes_stress": null,
  "passes_displacement": null,
  "overall_pass": null,
  "stress_hotspot_description": "",
  "notes": []
}
```

  → template file : /Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/CAD-Physics/code_base/fea_cad_one_sample/outputs/sample_00689964/04_manual_freecad_fea/fea_report.json


In [8]:
print("[STAGE] Validate report completion")
required_screenshots = [screenshots_dir / name for name in ["mesh.png", "fixed_region.png", "load_region.png", "von_mises.png", "displacement.png"]]
validation = api.validate_manual_fea_completion(manual_report, required_screenshots)
print(f"  → complete : {validation['is_complete']}")
print(f"  → missing fields : {validation['missing_fields']}")
print(f"  → missing files  : {validation['missing_evidence_paths']}")
if validation["is_complete"]:
    report_path.write_text(json.dumps(manual_report, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    print("  ✓ fea_report.json written")
else:
    print("  ⛔ blocked until required numeric fields and screenshots are provided")
for path in required_screenshots:
    if path.exists():
        display(Markdown(f"**{path.name}**"))
        display(Image(filename=str(path)))


src.fea.manual_report | INFO | validate_manual_fea_completion | result | is_complete=False | missing_fields=['max_von_mises_pa', 'max_displacement_mm', 'computed_safety_factor', 'passes_stress', 'passes_displacement', 'overall_pass', 'stress_hotspot_description'] | missing_evidence_paths=['/Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/CAD-Physics/code_base/fea_cad_one_sample/outputs/sample_00689964/04_manual_freecad_fea/screenshots/mesh.png', '/Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/CAD-Physics/code_base/fea_cad_one_sample/outputs/sample_00689964/04_manual_freecad_fea/screenshots/fixed_region.png', '/Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/CAD-Physics/code_base/fea_cad_one_sample/outputs/sample_00689964/04_manual_freecad_fea/screenshots/load_region.png', '/Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/CAD-Physics/code_base/fea_cad_one_sample/outp

[STAGE] Validate report completion
  → complete : False
  → missing fields : ['max_von_mises_pa', 'max_displacement_mm', 'computed_safety_factor', 'passes_stress', 'passes_displacement', 'overall_pass', 'stress_hotspot_description']
  → missing files  : ['/Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/CAD-Physics/code_base/fea_cad_one_sample/outputs/sample_00689964/04_manual_freecad_fea/screenshots/mesh.png', '/Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/CAD-Physics/code_base/fea_cad_one_sample/outputs/sample_00689964/04_manual_freecad_fea/screenshots/fixed_region.png', '/Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/CAD-Physics/code_base/fea_cad_one_sample/outputs/sample_00689964/04_manual_freecad_fea/screenshots/load_region.png', '/Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/CAD-Physics/code_base/fea_cad_one_sample/outputs/sample_00689964/04_manual_freec

## What this notebook proves

- The user has a clear FreeCAD FEM workflow.
- The manual FEA report template is ready for real results.
- The notebook can tell when post-FEA revision is blocked.
